In [ ]:
import pandas as pd
import numpy as np
import sys
import os

# Add parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.append(parent_dir)
from package_files.benefits_defns import *

# path = input("Please enter the input file path: ")
path = '../data/us_10m_nointernship_2018_2024_benefits.parquet.gzip'
print("reading data")
if path[-3:] == 'csv:':
    data = pd.read_csv(path)
elif path[-3:] == 'zip':
    data = pd.read_parquet(path)

In [ ]:
occupations_select = ['Architecture and Engineering Occupations','Arts, Design, Entertainment, Sports, and Media Occupations','Business and Financial Operations Occupations','Community and Social Service Occupations','Computer and Mathematical Occupations',
'Educational Instruction and Library Occupations',
'Healthcare Practitioners and Technical Occupations',
'Legal Occupations',
'Life, Physical, and Social Science Occupations',
'Management Occupations', 
'Office and Administrative Support Occupations',
'Personal Care and Service Occupations', 'Production Occupations',
'Sales and Related Occupations',
'Transportation and Material Moving Occupations',
]

In [ ]:
data_select = data[data[occupation].isin(occupations_select)]

In [ ]:
for (occ, year), group in data_select.groupby([occupation, 'YEAR']):
    print(occ, year)
    print(group)

In [ ]:
balanced_sample = []

for (occ, year), group in data_select.groupby([occupation, 'YEAR']):
    # Separate AI and non-AI roles
    ai_roles = group[group['AI ROLE'] == 1]
    non_ai_roles = group[group['AI ROLE'] == 0]
    
    # Determine the smaller size for sample size
    sample_size = min(len(ai_roles), len(non_ai_roles))
    
    # Random sample
    sampled_ai = ai_roles.sample(n=sample_size, random_state=42)
    sampled_non_ai = non_ai_roles.sample(n=sample_size, random_state=42)
    
    # Combine samples
    balanced_group = pd.concat([sampled_ai, sampled_non_ai])
    balanced_sample.append(balanced_group)

# Combine all balanced groups into a single DataFrame
final_balanced_sample = pd.concat(balanced_sample)

# Reset index for the final DataFrame
final_balanced_sample = final_balanced_sample.reset_index(drop=True)

# Output the balanced sample
final_balanced_sample


In [ ]:
ai_jobs_balanced = final_balanced_sample[final_balanced_sample['AI ROLE'] == 1]

# Group by occupation and YEAR
grouped = ai_jobs_balanced.groupby([occupation, 'YEAR'])

# Initialize an empty list to store results
results = []

# Loop through each group
for (occ, year), group in grouped:
    # Calculate the percentage of AI jobs with each benefit
    benefit_percentages = group[benefits4].mean() * 100  # Mean gives percentage for binary columns
    benefit_percentages['occupation'] = occ
    benefit_percentages['YEAR'] = year
    
    # Append to results
    results.append(benefit_percentages)

# Combine all results into a single DataFrame
benefits_summary_balanced = pd.DataFrame(results)

# Reset the index for readability
benefits_summary_balanced = benefits_summary_balanced.reset_index(drop=True)


In [ ]:
benefits_summary_balanced

In [ ]:
ai_jobs = final_balanced_sample[final_balanced_sample['AI ROLE'] == 1]
non_ai_jobs = final_balanced_sample[final_balanced_sample['AI ROLE'] == 0]

# Group by occupation and YEAR and sum benefits for AI and non-AI jobs
ai_counts = ai_jobs.groupby([occupation, 'YEAR'])[benefits4].sum().reset_index()
non_ai_counts = non_ai_jobs.groupby([occupation, 'YEAR'])[benefits4].sum().reset_index()

# Merge the counts for AI and non-AI jobs
merged_counts = ai_counts.merge(non_ai_counts, on=[occupation, 'YEAR'], suffixes=('_ai', '_non_ai'))

# Calculate the difference between AI and non-AI jobs for each benefit
for benefit in benefits4:
    merged_counts[f'{benefit}_difference'] = merged_counts[f'{benefit}_ai'] - merged_counts[f'{benefit}_non_ai']

# Select only the columns with differences and occupation-year identifiers
difference_summary = merged_counts[[occupation, 'YEAR'] + [f'{benefit}_difference' for benefit in benefits4]]

In [ ]:
difference_summary

In [ ]:
difference_summary.to_csv('../exports/occ_year_data/balanced_sample_diffs.csv', index=False)

In [ ]:
pd.set_option('display.max_rows', 100)

In [ ]:
final_balanced_sample.groupby([occupation, 'YEAR'])[benefits4].sum()